### Wave G — Marketing ROI / spend

Sample columns: Source/Channel, Spend, Leads, CPL, Closures, Cost per closure.

| Sheet | Yellow box |
|-------|------------|
| `Marketing_ROI` | #35, #36, #39, #40 |
| `Spend_By_Campaign` | #38 |



In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import ANALYSIS_START, ANALYSIS_END, START_TS, END_TS, CLOSURE_STAGE, q, write_excel

ADS = "leads_analytics"
d0, d1 = ANALYSIS_START.isoformat(), (ANALYSIS_END).isoformat()

def ads(sql):
    return q(sql, database=ADS)

# ponytail: column names from prior working pull; if schema drifts, fix here only
parts = []
try:
    parts.append(ads(f"""
    SELECT 'Google' AS platform, COALESCE(NULLIF(TRIM(CAST(campaign_name AS varchar)), ''), CAST(campaign_id AS varchar)) AS campaign,
           CAST(date AS date) AS day, SUM(TRY_CAST(amount_spent AS double)) AS spend
    FROM google_ads_raw
    WHERE CAST(date AS date) >= DATE '{d0}' AND CAST(date AS date) <= DATE '{d1}'
    GROUP BY 1,2,3
    """))
except Exception as e:
    print("google skip", e)
try:
    parts.append(ads(f"""
    SELECT 'Facebook' AS platform, COALESCE(NULLIF(TRIM(CAST(campaign_name AS varchar)), ''), CAST(campaign_id AS varchar)) AS campaign,
           CAST(date AS date) AS day, SUM(TRY_CAST(spend AS double)) AS spend
    FROM facebook_ads_raw
    WHERE CAST(date AS date) >= DATE '{d0}' AND CAST(date AS date) <= DATE '{d1}'
    GROUP BY 1,2,3
    """))
except Exception as e:
    print("facebook skip", e)
try:
    parts.append(ads(f"""
    SELECT 'Yoptima' AS platform, COALESCE(NULLIF(TRIM(CAST(campaign_name AS varchar)), ''), 'Yoptima') AS campaign,
           CAST(date AS date) AS day, SUM(TRY_CAST(amount_spent AS double)) AS spend
    FROM yoptima_ads_reports_raw
    WHERE CAST(date AS date) >= DATE '{d0}' AND CAST(date AS date) <= DATE '{d1}'
    GROUP BY 1,2,3
    """))
except Exception as e:
    print("yoptima skip", e)

spend = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["platform","campaign","day","spend"])
spend["spend"] = pd.to_numeric(spend["spend"], errors="coerce").fillna(0)

leads = q(f"""
WITH ct AS (
  SELECT CAST(cid AS varchar) cid, MAX(CAST(source AS varchar)) src FROM campaigntrackers WHERE cid IS NOT NULL GROUP BY 1
)
SELECT CAST(l._id AS varchar) lead_oid,
       COALESCE(NULLIF(TRIM(CAST(us.name AS varchar)), ''), TRIM(CAST(ls.name AS varchar))) channel
FROM leads l
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar)=CAST(l.bornsource AS varchar)
LEFT JOIN ct ON ct.cid=TRIM(CAST(l.utm_campaign AS varchar))
LEFT JOIN utmsources us ON CAST(us._id AS varchar)=ct.src
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")
closed = set(q(f"SELECT CAST(lead AS varchar) lead_oid FROM leadstagehistories WHERE CAST(stage AS varchar)='{CLOSURE_STAGE}' AND createdat >= {START_TS} AND createdat < {END_TS} GROUP BY 1")["lead_oid"])
leads["is_closed"] = leads["lead_oid"].isin(closed).astype(int)

def match(pat):
    m = leads["channel"].str.contains(pat, case=False, na=False)
    return int(m.sum()), int(leads.loc[m, "is_closed"].sum())

rows = []
for plat, pat in [("Google", r"Google|gads|google"), ("Facebook", r"Facebook|meta|fb"), ("Yoptima", r"Yoptima")]:
    n, c = match(pat)
    sp = float(spend.loc[spend["platform"]==plat, "spend"].sum())
    rows.append({"Source/Channel": plat, "Total Spend": round(sp, 2), "Leads": n, "Closures": c,
                 "CPL": round(sp/n, 2) if n else None, "Cost per closure": round(sp/c, 2) if c else None})
roi = pd.DataFrame(rows)
camp = spend.groupby(["platform","campaign"], as_index=False).agg(**{"Total Spend":("spend","sum")}).sort_values("Total Spend", ascending=False)
write_excel(ROOT / "Marketing_ROI_Spend_Analysis.xlsx", {"Marketing_ROI": roi, "Spend_By_Campaign": camp})
print("total spend", float(spend["spend"].sum()))

